In [ ]:
# !pip install vectorbt yfinance TA-Lib python-dateutil scipy

In [ ]:
import os
if not os.path.exists('/content/trading-strategies'):
    !git clone https://github.com/r-giov/trading-strategies.git /content/trading-strategies
os.chdir('/content/trading-strategies')

In [ ]:
import numpy as np
import pandas as pd
import vectorbt as vbt
import talib
import matplotlib.pyplot as plt
from scipy import stats
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

print("Imports ready.")

# Walk-Forward Validator
- Takes a strategy + params and runs rolling walk-forward validation
- Train on N months, test on M months, slide window forward
- Reports per-window Sharpe, return, drawdown, trade count
- Aggregates into a "Walk-Forward Efficiency Ratio" (OOS Sharpe / IS Sharpe across windows)
- Monte Carlo FTMO simulation on OOS-only returns
- Answers: "Would this strategy have worked across different market regimes?"

In [ ]:
# CONFIGURATION

TICKER = 'BTC-USD'
START_DATE = '2018-01-01'
TIMEFRAME = 'D'  # 'D', '1h', '4h'
INIT_CASH = 100_000
FEES = 0.0005
SLIPPAGE = 0.0005

# Strategy to validate — change these to test different strategies
STRATEGY = 'macd'  # 'macd', 'donchian', 'rsi_mr', 'triple_ema', 'stochastic'

# Fixed params (from grid search results)
PARAMS = {'fast': 30, 'slow': 59, 'signal': 6}

# Walk-forward settings
TRAIN_MONTHS = 6     # train on 6 months
TEST_MONTHS = 2      # test on 2 months
STEP_MONTHS = 2      # slide forward by 2 months (non-overlapping OOS)
MIN_TRADES_PER_WINDOW = 5

# Re-optimization mode
REOPTIMIZE = False   # If True, re-runs grid search each window (slow but realistic)
REOPT_GRID = {       # Only used if REOPTIMIZE = True
    'macd': {'fast': range(8, 40, 4), 'slow': range(20, 70, 5), 'signal': range(3, 15, 3)},
    'donchian': {'entry': range(5, 40, 5), 'exit': range(3, 20, 3), 'filter': range(10, 80, 10)},
}

# FTMO Monte Carlo
N_SIMULATIONS = 10_000

print(f"Strategy: {STRATEGY.upper()}")
print(f"Params: {PARAMS}")
print(f"Walk-forward: {TRAIN_MONTHS}m train / {TEST_MONTHS}m test / {STEP_MONTHS}m step")
print(f"Re-optimize each window: {REOPTIMIZE}")

In [ ]:
# DATA DOWNLOAD

import sys
sys.path.insert(0, 'lib')

try:
    from data_manager import DataManager
    dm = DataManager()
    data = dm.get(TICKER, start=START_DATE, timeframe=TIMEFRAME)
    print(f"Data loaded via DataManager: {len(data)} bars")
except Exception:
    import yfinance as yf
    if TIMEFRAME == 'D':
        data = yf.download(TICKER, start=START_DATE, auto_adjust=True)
    else:
        interval_map = {'1h': '1h', '4h': '4h', '15m': '15m'}
        data = yf.download(TICKER, start=START_DATE, interval=interval_map.get(TIMEFRAME, '1d'), auto_adjust=True)
    print(f"Data loaded via yfinance: {len(data)} bars")

# Handle MultiIndex columns
if isinstance(data.columns, pd.MultiIndex):
    data.columns = data.columns.get_level_values(0)

close = data['Close'].squeeze()
high = data['High'].squeeze()
low = data['Low'].squeeze()

print(f"Date range: {close.index[0].strftime('%Y-%m-%d')} to {close.index[-1].strftime('%Y-%m-%d')}")
print(f"Total bars: {len(close)}")

In [ ]:
# SIGNAL GENERATORS (all with 1-bar execution delay)

def macd_signals(close, fast, slow, signal):
    ml, sl, _ = talib.MACD(close.values.astype(float), fastperiod=fast, slowperiod=slow, signalperiod=signal)
    macd_s = pd.Series(ml, index=close.index)
    sig_s = pd.Series(sl, index=close.index)
    entries_raw = (macd_s.shift(1) <= sig_s.shift(1)) & (macd_s > sig_s)
    exits_raw = (macd_s.shift(1) >= sig_s.shift(1)) & (macd_s < sig_s)
    entries = pd.Series(np.where(entries_raw.shift(1).isna(), False, entries_raw.shift(1)), index=close.index, dtype=bool)
    exits = pd.Series(np.where(exits_raw.shift(1).isna(), False, exits_raw.shift(1)), index=close.index, dtype=bool)
    return entries, exits

def donchian_signals(close, high, low, entry_period, exit_period, filter_period):
    upper = pd.Series(talib.MAX(high.values, entry_period), index=close.index).shift(1)
    lower = pd.Series(talib.MIN(low.values, exit_period), index=close.index).shift(1)
    trend = pd.Series(talib.SMA(close.values, filter_period), index=close.index).shift(1)
    entries_raw = (close > upper) & (close > trend)
    exits_raw = (close < lower)
    entries = pd.Series(np.where(entries_raw.shift(1).isna(), False, entries_raw.shift(1)), index=close.index, dtype=bool)
    exits = pd.Series(np.where(exits_raw.shift(1).isna(), False, exits_raw.shift(1)), index=close.index, dtype=bool)
    return entries, exits

def rsi_mr_signals(close, rsi_len, oversold, overbought):
    rsi = pd.Series(talib.RSI(close.values, timeperiod=rsi_len), index=close.index)
    entries_raw = (rsi.shift(1) <= oversold) & (rsi > oversold)
    exits_raw = (rsi.shift(1) <= overbought) & (rsi > overbought)
    entries = pd.Series(np.where(entries_raw.shift(1).isna(), False, entries_raw.shift(1)), index=close.index, dtype=bool)
    exits = pd.Series(np.where(exits_raw.shift(1).isna(), False, exits_raw.shift(1)), index=close.index, dtype=bool)
    return entries, exits

def triple_ema_signals(close, ema1_period, ema2_period, ema3_period):
    e1 = pd.Series(talib.EMA(close.values.astype(float), ema1_period), index=close.index)
    e2 = pd.Series(talib.EMA(close.values.astype(float), ema2_period), index=close.index)
    e3 = pd.Series(talib.EMA(close.values.astype(float), ema3_period), index=close.index)
    entries_raw = ((e1.shift(1) <= e2.shift(1)) & (e1 > e2)) | ((e1.shift(1) <= e3.shift(1)) & (e1 > e3)) | ((e2.shift(1) <= e3.shift(1)) & (e2 > e3))
    exits_raw = ((e1.shift(1) >= e2.shift(1)) & (e1 < e2)) | ((e1.shift(1) >= e3.shift(1)) & (e1 < e3)) | ((e2.shift(1) >= e3.shift(1)) & (e2 < e3))
    entries = pd.Series(np.where(entries_raw.shift(1).isna(), False, entries_raw.shift(1)), index=close.index, dtype=bool)
    exits = pd.Series(np.where(exits_raw.shift(1).isna(), False, exits_raw.shift(1)), index=close.index, dtype=bool)
    return entries, exits

def stochastic_signals(close, high, low, k_period, d_period, oversold, overbought):
    slowk, slowd = talib.STOCH(high.values, low.values, close.values,
                                fastk_period=k_period, slowk_period=d_period,
                                slowk_matype=0, slowd_period=d_period, slowd_matype=0)
    k = pd.Series(slowk, index=close.index)
    entries_raw = (k.shift(1) <= oversold) & (k > oversold)
    exits_raw = (k.shift(1) >= overbought) & (k < overbought)
    entries = pd.Series(np.where(entries_raw.shift(1).isna(), False, entries_raw.shift(1)), index=close.index, dtype=bool)
    exits = pd.Series(np.where(exits_raw.shift(1).isna(), False, exits_raw.shift(1)), index=close.index, dtype=bool)
    return entries, exits

def get_signals(strategy, close, high, low, params):
    """Dispatch to the right signal generator."""
    if strategy == 'macd':
        return macd_signals(close, params['fast'], params['slow'], params['signal'])
    elif strategy == 'donchian':
        return donchian_signals(close, high, low, params['entry'], params['exit'], params['filter'])
    elif strategy == 'rsi_mr':
        return rsi_mr_signals(close, params['rsi_len'], params['oversold'], params['overbought'])
    elif strategy == 'triple_ema':
        return triple_ema_signals(close, params['ema1'], params['ema2'], params['ema3'])
    elif strategy == 'stochastic':
        return stochastic_signals(close, high, low, params['k_period'], params['d_period'],
                                  params['oversold'], params['overbought'])
    else:
        raise ValueError(f"Unknown strategy: {strategy}")

print("Signal generators ready.")

In [ ]:
# WALK-FORWARD VALIDATION ENGINE

from dateutil.relativedelta import relativedelta

def run_walk_forward(strategy, params, close, high, low, 
                     train_months, test_months, step_months,
                     reoptimize=False, reopt_grid=None, min_trades=5):
    """
    Rolling walk-forward validation.
    Returns list of window results with IS and OOS metrics.
    """
    results = []
    start_date = close.index[0]
    end_date = close.index[-1]
    
    window_start = start_date
    window_num = 0
    
    while True:
        train_start = window_start
        train_end = train_start + relativedelta(months=train_months)
        test_start = train_end
        test_end = test_start + relativedelta(months=test_months)
        
        if test_end > end_date:
            break
        
        # Slice data
        train_mask = (close.index >= train_start) & (close.index < train_end)
        test_mask = (close.index >= test_start) & (close.index < test_end)
        
        train_close = close[train_mask]
        test_close = close[test_mask]
        train_high = high[train_mask] if high is not None else None
        test_high = high[test_mask] if high is not None else None
        train_low = low[train_mask] if low is not None else None
        test_low = low[test_mask] if low is not None else None
        
        if len(train_close) < 50 or len(test_close) < 20:
            window_start += relativedelta(months=step_months)
            continue
        
        # Determine params for this window
        window_params = params.copy()
        
        if reoptimize and reopt_grid and strategy in reopt_grid:
            # Re-optimize on training data
            grid = reopt_grid[strategy]
            param_names = list(grid.keys())
            param_values = [list(v) for v in grid.values()]
            best_sharpe = -np.inf
            best_params = params.copy()
            
            from itertools import product
            for combo in product(*param_values):
                cp = dict(zip(param_names, combo))
                if strategy == 'macd' and cp.get('fast', 0) >= cp.get('slow', 999):
                    continue
                try:
                    e, x = get_signals(strategy, train_close, train_high, train_low, cp)
                    pf = vbt.Portfolio.from_signals(close=train_close, entries=e, exits=x,
                                                    init_cash=INIT_CASH, fees=FEES, slippage=SLIPPAGE, freq=TIMEFRAME)
                    if len(pf.trades) >= min_trades:
                        s = float(pf.sharpe_ratio(freq=TIMEFRAME))
                        if not np.isnan(s) and s > best_sharpe:
                            best_sharpe = s
                            best_params = cp.copy()
                except Exception:
                    continue
            window_params = best_params
        
        # IS backtest
        try:
            is_entries, is_exits = get_signals(strategy, train_close, train_high, train_low, window_params)
            pf_is = vbt.Portfolio.from_signals(close=train_close, entries=is_entries, exits=is_exits,
                                               init_cash=INIT_CASH, fees=FEES, slippage=SLIPPAGE, freq=TIMEFRAME)
            is_sharpe = float(pf_is.sharpe_ratio(freq=TIMEFRAME))
            is_return = float(pf_is.total_return())
            is_maxdd = float(pf_is.max_drawdown())
            is_trades = len(pf_is.trades)
        except Exception:
            window_start += relativedelta(months=step_months)
            continue
        
        # OOS backtest
        try:
            oos_entries, oos_exits = get_signals(strategy, test_close, test_high, test_low, window_params)
            pf_oos = vbt.Portfolio.from_signals(close=test_close, entries=oos_entries, exits=oos_exits,
                                                init_cash=INIT_CASH, fees=FEES, slippage=SLIPPAGE, freq=TIMEFRAME)
            oos_sharpe = float(pf_oos.sharpe_ratio(freq=TIMEFRAME))
            oos_return = float(pf_oos.total_return())
            oos_maxdd = float(pf_oos.max_drawdown())
            oos_trades = len(pf_oos.trades)
            oos_daily_rets = pf_oos.returns().values.ravel()
        except Exception:
            window_start += relativedelta(months=step_months)
            continue
        
        results.append({
            'window': window_num,
            'train_start': train_start.strftime('%Y-%m-%d'),
            'train_end': train_end.strftime('%Y-%m-%d'),
            'test_start': test_start.strftime('%Y-%m-%d'),
            'test_end': test_end.strftime('%Y-%m-%d'),
            'params': str(window_params),
            'is_sharpe': is_sharpe, 'oos_sharpe': oos_sharpe,
            'is_return': is_return, 'oos_return': oos_return,
            'is_maxdd': is_maxdd, 'oos_maxdd': oos_maxdd,
            'is_trades': is_trades, 'oos_trades': oos_trades,
            'oos_daily_rets': oos_daily_rets,
        })
        
        window_num += 1
        window_start += relativedelta(months=step_months)
    
    return results

print("Walk-forward engine ready.")

In [ ]:
# RUN WALK-FORWARD VALIDATION

print("=" * 70)
print(f"WALK-FORWARD VALIDATION: {STRATEGY.upper()} on {TICKER}")
print(f"Params: {PARAMS}")
print(f"Windows: {TRAIN_MONTHS}m train / {TEST_MONTHS}m test / {STEP_MONTHS}m step")
print(f"Re-optimize: {REOPTIMIZE}")
print("=" * 70)

wf_results = run_walk_forward(
    STRATEGY, PARAMS, close, high, low,
    TRAIN_MONTHS, TEST_MONTHS, STEP_MONTHS,
    REOPTIMIZE, REOPT_GRID if REOPTIMIZE else None, MIN_TRADES_PER_WINDOW
)

wf_df = pd.DataFrame([{k: v for k, v in r.items() if k != 'oos_daily_rets'} for r in wf_results])

print(f"\n{len(wf_results)} walk-forward windows completed\n")
print(f"{'Window':>6} {'Train Period':>24} {'Test Period':>24} {'IS Sharpe':>10} {'OOS Sharpe':>10} {'OOS Ret':>10} {'OOS DD':>10} {'Trades':>7}")
print("-" * 101)
for _, r in wf_df.iterrows():
    print(f"{int(r['window']):>6} {r['train_start']}\u2192{r['train_end']} {r['test_start']}\u2192{r['test_end']} "
          f"{r['is_sharpe']:>10.3f} {r['oos_sharpe']:>10.3f} {r['oos_return']:>9.1%} {r['oos_maxdd']:>9.1%} {int(r['oos_trades']):>7}")

# Aggregate metrics
avg_is_sharpe = wf_df['is_sharpe'].mean()
avg_oos_sharpe = wf_df['oos_sharpe'].mean()
wfe = avg_oos_sharpe / avg_is_sharpe if abs(avg_is_sharpe) > 0.01 else 0  # Walk-Forward Efficiency
pct_profitable = (wf_df['oos_return'] > 0).sum() / len(wf_df) * 100
avg_oos_return = wf_df['oos_return'].mean()
worst_oos_dd = wf_df['oos_maxdd'].min()
total_oos_trades = wf_df['oos_trades'].sum()

print(f"\n{'='*70}")
print(f"WALK-FORWARD SUMMARY")
print(f"{'='*70}")
print(f"  Windows:                {len(wf_results)}")
print(f"  Avg IS Sharpe:          {avg_is_sharpe:.3f}")
print(f"  Avg OOS Sharpe:         {avg_oos_sharpe:.3f}")
print(f"  Walk-Forward Efficiency:{wfe:.1%} {'(GOOD > 50%)' if wfe > 0.5 else '(WEAK < 50%)'}")
print(f"  % Profitable Windows:   {pct_profitable:.0f}%")
print(f"  Avg OOS Return:         {avg_oos_return:.1%}")
print(f"  Worst OOS Drawdown:     {worst_oos_dd:.1%}")
print(f"  Total OOS Trades:       {total_oos_trades}")
print(f"{'='*70}")

# Verdict
if wfe > 0.7 and pct_profitable >= 60:
    wf_verdict = "ROBUST \u2014 Strategy generalizes well across market regimes"
elif wfe > 0.4 and pct_profitable >= 50:
    wf_verdict = "MODERATE \u2014 Some regime dependence but generally viable"
elif wfe > 0.2:
    wf_verdict = "WEAK \u2014 Significant overfitting or regime sensitivity"
else:
    wf_verdict = "OVERFIT \u2014 Strategy does not generalize; avoid live trading"

print(f"\nVERDICT: {wf_verdict}")

In [ ]:
# MONTE CARLO FTMO — USING OOS-ONLY RETURNS

print("=" * 70)
print("MONTE CARLO FTMO \u2014 Walk-Forward OOS Returns Only")
print("=" * 70)

# Concatenate all OOS daily returns
all_oos_rets = np.concatenate([r['oos_daily_rets'] for r in wf_results])
all_oos_rets = all_oos_rets[~np.isnan(all_oos_rets)]

print(f"OOS daily returns: {len(all_oos_rets)} days")
print(f"  Mean:     {np.mean(all_oos_rets)*100:.4f}%")
print(f"  Std:      {np.std(all_oos_rets)*100:.4f}%")
print(f"  Skew:     {float(stats.skew(all_oos_rets)):.3f}")
print(f"  Kurtosis: {float(stats.kurtosis(all_oos_rets)):.3f}")

FTMO_ACCOUNT = 100_000
PROFIT_TARGET = 0.10
MAX_DAILY_LOSS = 0.05
MAX_TOTAL_LOSS = 0.10
TRADING_DAYS = 30

np.random.seed(42)
equity_paths = np.zeros((N_SIMULATIONS, TRADING_DAYS + 1))
equity_paths[:, 0] = FTMO_ACCOUNT

passed = np.zeros(N_SIMULATIONS, dtype=bool)
blown = np.zeros(N_SIMULATIONS, dtype=bool)
daily_blown = np.zeros(N_SIMULATIONS, dtype=bool)
final_equity = np.zeros(N_SIMULATIONS)

for sim in range(N_SIMULATIONS):
    sim_rets = np.random.choice(all_oos_rets, size=TRADING_DAYS, replace=True)
    eq = FTMO_ACCOUNT
    for day in range(TRADING_DAYS):
        day_start = eq
        eq *= (1 + sim_rets[day])
        equity_paths[sim, day + 1] = eq
        if (eq - day_start) / FTMO_ACCOUNT < -MAX_DAILY_LOSS:
            daily_blown[sim] = True; equity_paths[sim, day+2:] = eq; break
        if (eq - FTMO_ACCOUNT) / FTMO_ACCOUNT < -MAX_TOTAL_LOSS:
            blown[sim] = True; equity_paths[sim, day+2:] = eq; break
        if (eq - FTMO_ACCOUNT) / FTMO_ACCOUNT >= PROFIT_TARGET:
            passed[sim] = True; equity_paths[sim, day+2:] = eq; break
    final_equity[sim] = equity_paths[sim, -1]

n_passed = passed.sum()
n_blown_total = blown.sum()
n_blown_daily = daily_blown.sum()
n_survived = N_SIMULATIONS - n_passed - n_blown_total - n_blown_daily
pass_rate = n_passed / N_SIMULATIONS * 100

print(f"\nMonte Carlo Results ({N_SIMULATIONS:,} sims using OOS-only returns):")
print(f"  PASSED:        {n_passed:>6,} ({pass_rate:.1f}%)")
print(f"  Blown (total): {n_blown_total:>6,} ({n_blown_total/N_SIMULATIONS*100:.1f}%)")
print(f"  Blown (daily): {n_blown_daily:>6,} ({n_blown_daily/N_SIMULATIONS*100:.1f}%)")
print(f"  Still trading: {n_survived:>6,} ({n_survived/N_SIMULATIONS*100:.1f}%)")
print(f"  Median equity: ${np.median(final_equity):,.0f}")

if pass_rate >= 50: mc_verdict = f"FAVORABLE \u2014 {pass_rate:.1f}%"
elif pass_rate >= 25: mc_verdict = f"POSSIBLE \u2014 {pass_rate:.1f}%"
elif pass_rate >= 10: mc_verdict = f"CHALLENGING \u2014 {pass_rate:.1f}%"
else: mc_verdict = f"UNLIKELY \u2014 {pass_rate:.1f}%"
print(f"\nFTMO Verdict (OOS-only): {mc_verdict}")

In [ ]:
# VISUALIZATION

fig, axes = plt.subplots(3, 2, figsize=(18, 18))
fig.suptitle(f'Walk-Forward Validation: {STRATEGY.upper()} on {TICKER} \u2014 {PARAMS}',
             fontsize=16, fontweight='bold')

# (0,0) IS vs OOS Sharpe per window
x = range(len(wf_df))
axes[0, 0].bar([i - 0.15 for i in x], wf_df['is_sharpe'], width=0.3, color='#1f77b4', label='IS Sharpe', alpha=0.8)
axes[0, 0].bar([i + 0.15 for i in x], wf_df['oos_sharpe'], width=0.3, color='#ff7f0e', label='OOS Sharpe', alpha=0.8)
axes[0, 0].axhline(0, color='black', linewidth=1)
axes[0, 0].axhline(avg_oos_sharpe, color='red', linestyle='--', linewidth=1.5, label=f'Avg OOS: {avg_oos_sharpe:.3f}')
axes[0, 0].set_xlabel('Window')
axes[0, 0].set_ylabel('Sharpe Ratio')
axes[0, 0].set_title('IS vs OOS Sharpe per Window', fontsize=13, fontweight='bold')
axes[0, 0].legend(fontsize=9)
axes[0, 0].grid(axis='y', alpha=0.3)

# (0,1) OOS return per window
colors = ['#2ca02c' if r > 0 else '#d62728' for r in wf_df['oos_return']]
axes[0, 1].bar(x, wf_df['oos_return'] * 100, color=colors, edgecolor='black', linewidth=0.5)
axes[0, 1].axhline(0, color='black', linewidth=1)
axes[0, 1].set_xlabel('Window')
axes[0, 1].set_ylabel('OOS Return (%)')
axes[0, 1].set_title(f'OOS Return per Window ({pct_profitable:.0f}% profitable)', fontsize=13, fontweight='bold')
axes[0, 1].grid(axis='y', alpha=0.3)

# (1,0) Walk-forward efficiency over time
cumulative_wfe = []
for i in range(1, len(wf_df) + 1):
    avg_is = wf_df['is_sharpe'].iloc[:i].mean()
    avg_oos = wf_df['oos_sharpe'].iloc[:i].mean()
    cumulative_wfe.append(avg_oos / avg_is if abs(avg_is) > 0.01 else 0)
axes[1, 0].plot(range(len(cumulative_wfe)), [w * 100 for w in cumulative_wfe], 
                color='#1f77b4', linewidth=2, marker='o', markersize=5)
axes[1, 0].axhline(50, color='green', linestyle='--', alpha=0.5, label='50% (good)')
axes[1, 0].axhline(0, color='red', linestyle='--', alpha=0.5, label='0% (no edge)')
axes[1, 0].set_xlabel('Window (cumulative)')
axes[1, 0].set_ylabel('WF Efficiency (%)')
axes[1, 0].set_title(f'Walk-Forward Efficiency Ratio: {wfe:.1%}', fontsize=13, fontweight='bold')
axes[1, 0].legend(fontsize=9)
axes[1, 0].grid(True, alpha=0.3)

# (1,1) OOS drawdown per window
axes[1, 1].bar(x, wf_df['oos_maxdd'] * 100, color='#d62728', alpha=0.7, edgecolor='black', linewidth=0.5)
axes[1, 1].axhline(worst_oos_dd * 100, color='red', linestyle='--', linewidth=1.5, 
                    label=f'Worst: {worst_oos_dd:.1%}')
axes[1, 1].set_xlabel('Window')
axes[1, 1].set_ylabel('Max Drawdown (%)')
axes[1, 1].set_title('OOS Max Drawdown per Window', fontsize=13, fontweight='bold')
axes[1, 1].legend(fontsize=9)
axes[1, 1].grid(axis='y', alpha=0.3)

# (2,0) Monte Carlo equity paths
sample_idx = np.random.choice(N_SIMULATIONS, min(200, N_SIMULATIONS), replace=False)
for i in sample_idx:
    c = '#2ca02c' if passed[i] else ('#d62728' if (blown[i] or daily_blown[i]) else '#aaaaaa')
    a = 0.4 if (passed[i] or blown[i] or daily_blown[i]) else 0.1
    axes[2, 0].plot(range(TRADING_DAYS + 1), equity_paths[i], color=c, alpha=a, linewidth=0.5)
axes[2, 0].axhline(FTMO_ACCOUNT * 1.1, color='green', linestyle='--', linewidth=2.5, label='Target')
axes[2, 0].axhline(FTMO_ACCOUNT * 0.9, color='red', linestyle='--', linewidth=2.5, label='Max Loss')
axes[2, 0].set_title(f'MC FTMO (OOS returns) \u2014 {pass_rate:.1f}% pass', fontsize=13, fontweight='bold')
axes[2, 0].set_xlabel('Trading Day')
axes[2, 0].set_ylabel('Equity ($)')
axes[2, 0].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))
axes[2, 0].legend(fontsize=9)
axes[2, 0].grid(True, alpha=0.2)

# (2,1) Summary stats box
summary_text = (
    f"WALK-FORWARD VALIDATION REPORT\n"
    f"{'='*40}\n"
    f"Strategy:         {STRATEGY.upper()}\n"
    f"Ticker:           {TICKER}\n"
    f"Params:           {PARAMS}\n"
    f"{'='*40}\n"
    f"Windows:          {len(wf_results)}\n"
    f"Avg IS Sharpe:    {avg_is_sharpe:.3f}\n"
    f"Avg OOS Sharpe:   {avg_oos_sharpe:.3f}\n"
    f"WF Efficiency:    {wfe:.1%}\n"
    f"% Profitable:     {pct_profitable:.0f}%\n"
    f"Avg OOS Return:   {avg_oos_return:.1%}\n"
    f"Worst OOS DD:     {worst_oos_dd:.1%}\n"
    f"Total OOS Trades: {total_oos_trades}\n"
    f"{'='*40}\n"
    f"FTMO MC Pass:     {pass_rate:.1f}%\n"
    f"MC Verdict:       {mc_verdict}\n"
    f"{'='*40}\n"
    f"WF Verdict:       {wf_verdict}"
)
axes[2, 1].text(0.05, 0.95, summary_text, transform=axes[2, 1].transAxes,
                fontsize=11, verticalalignment='top', fontfamily='monospace',
                bbox=dict(boxstyle='round,pad=0.8', facecolor='lightyellow', edgecolor='#333', alpha=0.9))
axes[2, 1].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# EXPORT + TELEGRAM

import json, os, urllib.request, urllib.parse

if os.path.exists('/content/drive/MyDrive'):
    export_root = '/content/drive/MyDrive/strategy_exports'
else:
    export_root = os.path.join(os.getcwd(), 'strategy_exports')

export_dir = os.path.join(export_root, 'Walk_Forward', f'{STRATEGY}_{TICKER}', 'latest')
os.makedirs(export_dir, exist_ok=True)

# Save window results
wf_df.to_csv(os.path.join(export_dir, 'walk_forward_windows.csv'), index=False)

# Save summary
summary = {
    'strategy': STRATEGY, 'ticker': TICKER, 'params': PARAMS,
    'timeframe': TIMEFRAME,
    'train_months': TRAIN_MONTHS, 'test_months': TEST_MONTHS, 'step_months': STEP_MONTHS,
    'reoptimize': REOPTIMIZE,
    'n_windows': len(wf_results),
    'avg_is_sharpe': avg_is_sharpe, 'avg_oos_sharpe': avg_oos_sharpe,
    'wf_efficiency': wfe,
    'pct_profitable_windows': pct_profitable,
    'avg_oos_return': avg_oos_return,
    'worst_oos_dd': worst_oos_dd,
    'total_oos_trades': int(total_oos_trades),
    'wf_verdict': wf_verdict,
    'mc_pass_rate': pass_rate,
    'mc_verdict': mc_verdict,
}
with open(os.path.join(export_dir, 'summary.json'), 'w') as f:
    json.dump(summary, f, indent=2, default=str)

print(f"Exported to: {export_dir}")

# Telegram
TG_BOT_TOKEN = os.getenv('TG_BOT_TOKEN', '8691594427:AAGKbcObikmFxr3yJk5kVkIFkIDAuVyqeoo')
TG_CHAT_ID = os.getenv('TG_CHAT_ID', '6451760231')

def send_telegram(message):
    url = f"https://api.telegram.org/bot{TG_BOT_TOKEN}/sendMessage"
    d = urllib.parse.urlencode({'chat_id': TG_CHAT_ID, 'text': message, 'parse_mode': 'Markdown'}).encode()
    try:
        req = urllib.request.Request(url, data=d)
        with urllib.request.urlopen(req, timeout=10) as resp:
            return resp.status == 200
    except Exception as e:
        print(f"Telegram failed: {e}")
        return False

msg = (
    f"*Walk-Forward Validation Complete*\n\n"
    f"Strategy: {STRATEGY.upper()} on {TICKER}\n"
    f"Params: {PARAMS}\n\n"
    f"*Results ({len(wf_results)} windows):*\n"
    f"  WF Efficiency: {wfe:.1%}\n"
    f"  Avg OOS Sharpe: {avg_oos_sharpe:.3f}\n"
    f"  Profitable: {pct_profitable:.0f}%\n"
    f"  Worst DD: {worst_oos_dd:.1%}\n\n"
    f"*FTMO MC (OOS returns):*\n"
    f"  Pass Rate: {pass_rate:.1f}%\n"
    f"  Verdict: {mc_verdict}\n\n"
    f"*WF Verdict: {wf_verdict}*"
)

if send_telegram(msg):
    print("Results sent to Telegram!")
else:
    print("Telegram failed.")
    print(msg)